In [2]:
import gdown
gdown.download(id="1u_XGZq4lODKq56OXp9ofXbwcW1Qx3vc0", output="dummy/unet.pth", quiet=False)

Downloading...
From (original): https://drive.google.com/uc?id=1u_XGZq4lODKq56OXp9ofXbwcW1Qx3vc0
From (redirected): https://drive.google.com/uc?id=1u_XGZq4lODKq56OXp9ofXbwcW1Qx3vc0&confirm=t&uuid=8b8edba5-b100-4b4c-8e78-c8e20fb8614c
To: /Users/mohamedmafaz/Desktop/Object Detection from scratch/dummy/unet.pth
100%|██████████| 214M/214M [00:11<00:00, 18.7MB/s]


'dummy/unet.pth'

In [13]:
import argparse

import torch
from torch import nn
import matplotlib.pyplot as plt
import numpy as np

from tqdm import tqdm

from torch.utils.data import Dataset, random_split, DataLoader, dataset

import pathlib
import matplotlib.patches as patches
from data.pets_dataset import OxfordIIITPetDataset, get_class_map

from models import MultiTaskPerceptionModel
from losses import DiceLoss, IoULoss

bce = nn.BCEWithLogitsLoss()

def unet_loss_fn(pred, target):
    return bce(pred, target) + DiceLoss()(pred, target)

def bbox_loss_fn(pred, target):
    iou_loss = IoULoss()
    return iou_loss(pred, target)


In [4]:
# parser = argparse.ArgumentParser(description="Inferencing Object detection", conflict_handler="resolve")


# parser.add_argument('-n_breeds', "--num_breeds", type=int, default=37, help="Total number of breeds to classify")
# parser.add_argument('-s_class', "--seg_classes", type=int, default=3, help="Total number of classes to segment")
# parser.add_argument('-in_c', "--in_channels", type=int, default=3, help="In channels of images")
# parser.add_argument('-c_path', "--classifier_path", type=str, default="checkpoints/classifier.pth", help="Path for classifier model (.pth)")
# parser.add_argument('-l_path', "--localizer_path", type=str, default="checkpoints/localizer.pth", help="Path for localizer model (.pth)")
# parser.add_argument('-u_path', "--unet_path", type=str, default="checkpoints/unet.pth", help="Path for unet model (.pth)")
# parser.add_argument('-d_path', "--dataset_path", type=str, default="oxford-iiit-pet", help="Path for dataset model")


# args = parser.parse_args()

In [ ]:
train_ratio = 0.8
BATCH_SIZE = 4
EPOCHS = 1

# num_breeds
# seg_classes
# in_channels

# # for fine tuning
# classifier_path
# localizer_path
# unet_path

# # for saving tuning
# classifier_path
# localizer_path
# unet_path

In [ ]:
multitask_model = MultiTaskPerceptionModel(num_breeds      = 37, 
                                           seg_classes     = 3, 
                                           in_channels     = 3, 
                                           classifier_path = None,
                                           localizer_path  = None,
                                           unet_path       = None)

classifier loaded!
localizer loaded!
unet loaded!


In [10]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [11]:
classifier = multitask_model.model_classifier.to(device)
localizer  = multitask_model.model_localizer.to(device)
unet       = multitask_model.unet.to(device)


In [ ]:
classifier_optimizer = torch.optim.Adam(classifier.parameters(), lr=1e-4, weight_decay=1e-4)
classifier_loss_fn   = nn.CrossEntropyLoss()

localizer_optimizer = torch.optim.Adam(localizer.parameters(), lr=1e-4, weight_decay=1e-4)
localizer_loss_fn = IoULoss()

unet_optimizer = torch.optim.Adam(localizer.parameters(), lr=1e-4, weight_decay=1e-4)
bce = nn.BCEWithLogitsLoss()
def loss_fn(pred, target):
    return bce(pred, target) + DiceLoss()(pred, target)

In [ ]:
mappings = get_class_map(pathlib.Path("oxford-iiit-pet"))

dataset = OxfordIIITPetDataset(root_dir = "oxford-iiit-pet")

train_ds, test_ds = random_split(dataset, [int(train_ratio * len(dataset)), len(dataset)-int(train_ratio * len(dataset))])

train_dl = DataLoader(train_ds, batch_size = BATCH_SIZE, shuffle=True)
test_dl  = DataLoader(test_ds,  batch_size = BATCH_SIZE, shuffle=False)


In [ ]:
train_loss_tracker = []
test_loss_tracker  = []
print("TRAINING CLASSIFIER")
for epoch in range(EPOCHS):
    # Train
    classifier.train()
    train_loss = 0.0

    for batch in tqdm(train_dl, desc=f"Epoch {epoch+1}/{EPOCHS} [train]"):
        images = batch['image'].to(device)
        class_ids = batch['class_id'].to(device)

        logits = classifier(images)
        loss = classifier_loss_fn(logits, class_ids)

        classifier_optimizer.zero_grad()
        loss.backward()
        classifier_optimizer.step()

        train_loss += loss.item()

    train_loss /= len(train_dl)
    train_loss_tracker.append(train_loss)

    # Test
    classifier.eval()
    test_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for batch in tqdm(test_dl, desc=f"Epoch {epoch+1}/{EPOCHS} [test] "):
            images = batch['image'].to(device)
            class_ids = batch['class_id'].to(device)

            logits = classifier(images)
            loss = classifier_loss_fn(logits, class_ids)
            test_loss += loss.item()

            preds = logits.argmax(dim=1)
            correct += (preds == class_ids).sum().item()
            total += class_ids.size(0)

    test_loss /= len(test_dl)
    test_loss_tracker.append(test_loss)
    test_acc = correct / total * 100

    print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {train_loss:.4f} | Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.2f}%")

In [ ]:
print("LOCALIZER CLASSIFIER")
for epoch in range(EPOCHS):
    # Train
    localizer.train()
    train_loss = 0.0
 
    for batch in tqdm(train_dl, desc=f"Epoch {epoch+1}/{EPOCHS} [train]"):
        images = batch['image'].to(device)
        class_ids = batch['class_id'].to(device)
        bbox = batch['bbox'].to(device)
 
        pred_bbox = localizer(images)
        iou = localizer_loss_fn(pred_bbox, bbox)
        l1  = torch.abs(pred_bbox - bbox).mean()
        loss = iou + 1.0 * l1 # l1

        localizer_optimizer.zero_grad()
        loss.backward()
        localizer_optimizer.step()
 
        train_loss += loss.item()
 
    train_loss /= len(train_dl)
    train_loss_tracker.append(train_loss)
 
    # Test
    localizer.eval()
    test_loss = 0.0
    correct = 0
    total = 0
 
    with torch.no_grad():
        for batch in tqdm(test_dl, desc=f"Epoch {epoch+1}/{EPOCHS} [test] "):
            images = batch['image'].to(device)
            class_ids = batch['class_id'].to(device)
            bbox = batch['bbox'].to(device)
 
            pred_bbox = localizer(images)
            iou = localizer_loss_fn(pred_bbox, bbox)
            l1  = torch.abs(pred_bbox - bbox).mean()
            loss = iou + 1.0 * l1 # l1

            test_loss += loss.item()
 
    test_loss /= len(test_dl)
    test_loss_tracker.append(test_loss)
 
    print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {train_loss:.4f} | Test Loss: {test_loss:.4f} ")

In [ ]:
print("UNET CLASSIFIER")
train_loss_tracker = []
test_loss_tracker = []
for epoch in range(EPOCHS):
    # Train
    unet.train()
    train_loss = 0.0
 
    for batch in tqdm(train_dl, desc=f"Epoch {epoch+1}/{EPOCHS} [train]"):
        images = batch['image'].to(device)
        masks = batch['mask'].to(device)
 
        pred_mask = unet(images)
        loss = unet_loss_fn(pred_mask, masks)

        unet_optimizer.zero_grad()
        loss.backward()
        unet_optimizer.step()
 
        train_loss += loss.item()
 
    train_loss /= len(train_dl)
    train_loss_tracker.append(train_loss)
 
    unet.eval()
    test_loss = 0.0
 
    with torch.no_grad():
        for batch in tqdm(test_dl, desc=f"Epoch {epoch+1}/{EPOCHS} [test] "):
          masks = batch['mask'].to(device)
          images = batch['image'].to(device)
          pred_mask = unet(images)
          loss = loss_fn(pred_mask, masks)

          test_loss += loss.item()
 
 
    test_loss /= len(test_dl)
    test_loss_tracker.append(test_loss)
 
    print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {train_loss:.4f} | Test Loss: {test_loss:.4f} ")